In [1]:
import os
import numpy as np
import pandas as pd
from scipy.io import loadmat
from datetime import datetime

from pynwb import NWBFile, NWBHDF5IO, TimeSeries

In [10]:
def load_mat_file(filepath):
    """Load MATLAB file (supports both v7.3 and older formats)."""
    
    try:
        # Try HDF5 (v7.3)
        return h5py.File(filepath, "r"), "h5py"
    
    except OSError:
        # Fall back to old MATLAB format
        return loadmat(filepath, squeeze_me=True, struct_as_record=False), "scipy"

In [6]:
mat=load_mat_file("/lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/for_BundleNet/5-2-22_w5_H_wbstructforPCA.mat")

In [7]:
print(mat.keys())

<KeysViewHDF5 ['#refs#', 'ID', 'ID1', 'ID2', 'ID3', 'MT', 'added', 'blobThreads', 'blobThreads_sorted', 'dateRan', 'deltaFOverF', 'deltaFOverFNoBackSub', 'deltaFOverF_bc', 'deltaFOverF_bc_options', 'deltaFOverF_bc_suppData', 'displayname', 'exclusionList', 'f0', 'f_background', 'f_bonded', 'f_parents', 'fps', 'globalDriftX', 'globalDriftY', 'mask', 'mask_background', 'mask_nooverlap', 'metadata', 'neuronlookup', 'nn', 'numZ', 'nx', 'ny', 'nz', 'options', 'replacements', 'simple', 'stimulus', 'tblobs', 'totalTime', 'trialname', 'tv']>


In [12]:
def extract_data(mat, loader_type):

    if loader_type == "h5py":
        # --- v7.3 ---
        calcium_activity = mat["deltaFOverF_bc"][:]
        neuron_ids = mat["ID"][:]
        stimulus = mat["stimulus"]["switchtimes"][:]
        state_annotation = mat["simple"]["traceColoring"][:]
        fps = float(mat["fps"][()])

    else:
        # --- old MATLAB ---
        calcium_activity = mat["deltaFOverF_bc"]
        neuron_ids = mat["ID"]
        
        stimulus = mat["stimulus"].switchtimes if hasattr(mat["stimulus"], "switchtimes") else None
        state_annotation = mat["simple"].traceColoring if hasattr(mat["simple"], "traceColoring") else None
        
        fps = float(mat["fps"])

    # --- Convert to numpy ---
    calcium_activity = np.array(calcium_activity)

    # Fix orientation
    if calcium_activity.shape[0] < calcium_activity.shape[1]:
        calcium_activity = calcium_activity.T

    time = np.arange(calcium_activity.shape[0]) / fps

    neuron_ids = np.array(neuron_ids).squeeze()
    stimulus = np.array(stimulus).squeeze() if stimulus is not None else None
    state_annotation = np.array(state_annotation).squeeze() if state_annotation is not None else None

    return calcium_activity, time, neuron_ids, stimulus, state_annotation


def create_nwb(calcium_activity, time, neuron_ids, stimulus, state_annotation, file_id):
    """Create NWB structure."""

    nwbfile = NWBFile(
        session_description="Converted from MATLAB rawdata",
        identifier=file_id,
        session_start_time=datetime.now()
    )

    # ---- Add main signal ----
    if calcium_activity is not None:
        ts = TimeSeries(
            name="signal",
            data=calcium_activity,
            timestamps=time,
            unit="a.u."
        )
        nwbfile.add_acquisition(ts)

    # Neuron IDs → Units
    if neuron_ids is not None:
        nwbfile.add_unit_column("neuron_id", "Original neuron ID")

        for i, nid in enumerate(neuron_ids):
            nwbfile.add_unit(id=i, neuron_id=str(nid))

    # Stimulus
    if stimulus is not None:
        stim_ts = TimeSeries(
            name="stimulus",
            data=stimulus,
            timestamps=time,
            unit="a.u."
        )
        nwbfile.add_stimulus(stim_ts)

    # ---- Add behavior ----
    if state_annotation is not None:
        ts_beh = TimeSeries(
            name="behavior_state",
            data=state_annotation,
            timestamps=time,
            unit="category"
        )
        nwbfile.add_acquisition(ts_beh)

    return nwbfile


def convert_folder(input_folder, output_folder):
    """Convert all .mat files in a folder to NWB."""

    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if not filename.endswith(".mat"):
            continue

        filepath = os.path.join(input_folder, filename)
        print(f"Processing: {filename}")
        print(h5py.is_hdf5(filepath))
        mat, filetype = load_mat_file(filepath)

        data, time,  neuron_ids, stimulus, behavior = extract_data(mat, filetype)

        file_id = os.path.splitext(filename)[0]

        nwbfile = create_nwb(data, time,  neuron_ids, stimulus, behavior, file_id)

        output_path = os.path.join(output_folder, file_id + ".nwb")

        with NWBHDF5IO(output_path, "w") as io:
            io.write(nwbfile)

        print(f"Saved: {output_path}")


if __name__ == "__main__":
    input_folder = "/lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/matfiles"
    output_folder = "/lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles"

    convert_folder(input_folder, output_folder)

Processing: 13-10-21_w2_T_wbstructforPCA.mat
False


/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/pynwb/file.py:468: UserWarning: Date is missing timezone information. Updating to local timezone.
  args_to_set['session_start_time'] = _add_missing_timezone(session_start_time)
/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/pynwb/base.py:203: UserWarning: TimeSeries 'stimulus': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn("%s '%s': Length of data does not match length of timestamps. Your data may be transposed. "


Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/13-10-21_w2_T_wbstructforPCA.nwb
Processing: 9-10-21_w3_T_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/9-10-21_w3_T_wbstructforPCA.nwb
Processing: 13-10-21_w2_H_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/13-10-21_w2_H_wbstructforPCA.nwb
Processing: 13-10-21_w3_H_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/13-10-21_w3_H_wbstructforPCA.nwb
Processing: 5-2-22_w5_T_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/5-2-22_w5_T_wbstructforPCA.nwb
Processing: 5-2-22_w9_T_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/5-2-22_w9_T_wbstructforPCA.nwb
Processing: 5-2-2

/lisc/data/scratch/neurobiology/zimmer/jalaja/conda/envs/jupyter/lib/python3.8/site-packages/pynwb/base.py:203: UserWarning: TimeSeries 'behavior_state': Length of data does not match length of timestamps. Your data may be transposed. Time should be on the 0th dimension
  warn("%s '%s': Length of data does not match length of timestamps. Your data may be transposed. "


Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/5-2-22_w5_H_wbstructforPCA.nwb
Processing: 5-2-22_w9_H_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/5-2-22_w9_H_wbstructforPCA.nwb
Processing: 13-10-21_w4_H_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/13-10-21_w4_H_wbstructforPCA.nwb
Processing: 9-2-22_w7_T_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/9-2-22_w7_T_wbstructforPCA.nwb
Processing: 9-2-22_w7_H_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/9-2-22_w7_H_wbstructforPCA.nwb
Processing: 9-10-21_w1_H_wbstructforPCA.mat
False
Saved: /lisc/data/scratch/neurobiology/zimmer/jalaja/imaging/newline/cleaned/nwb/nwbfiles/9-10-21_w1_H_wbstructforPCA.nwb
Processing: 13-10-21_w3